In [ ]:
from bs4 import BeautifulSoup as bs
import requests
from tqdm import tqdm
tqdm.pandas()
import pandas as pd
import time

In [ ]:
import pandas as pd

# Carica il file Excel (specifica il nome del file e il foglio corretto)
df = pd.read_excel('Unione_Recensioni_Lingua.xlsx', sheet_name='Sheet1')

# Visualizza i primi commenti
print(df.head())

   Unnamed: 0               Author  \
0           0           SanyiAntek   
1           1                 drea   
2           2               Hannah   
3           3            cinegallo   
4           4  Jack Anderson Keane   

                                              Review                 Date  \
0  - "Do you like Mozart?"- "I like Depeche Mode....  2021-02-15 00:00:00   
1  - "how long do i have?" - "i'm glad you asked....          29 Sep 2024   
2  - "I got my eye on you"- "I told you I don't w...          26 Jun 2022   
3  - "I present you, Blankman, the Harris Award f...          29 Sep 2022   
4  - "I was with our illustrious creator, Mr. Wey...          02 Oct 2017   

   Likes             Film lingua  
0    9.0  ex-machina-2015     en  
1    1.0          why-him     en  
2    5.0       iron-man-2     en  
3    3.0         blankman     en  
4   23.0   alien-covenant     en  


In [ ]:
from transformers import pipeline

# Carica il modello di Sentiment Analysis
sentiment_analyzer = pipeline('sentiment-analysis')

# Applica l'analisi del sentiment a ciascun commento
df['Sentiment'] = df['Testo'].apply(lambda comment: sentiment_analyzer(comment)[0]['label'])
df['Sentiment_Score'] = df['Testo'].apply(lambda comment: sentiment_analyzer(comment)[0]['score'])

# Visualizza i risultati
print(df[['Testo', 'Sentiment', 'Sentiment_Score']].head())


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


KeyError: 'Testo'

In [ ]:
# Carica il modello di Emotion Analysis
emotion_analyzer = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=True)

# Funzione per estrarre le emozioni principali da ciascun commento
def get_emotions(comment):
    emotions = emotion_analyzer(comment)
    # Restituisce solo l'emozione con il punteggio più alto
    return max(emotions[0], key=lambda x: x['score'])['label']

# Applica l'analisi delle emozioni a ciascun commento
df['Emotion'] = df['Testo'].apply(get_emotions)

# Visualizza i risultati
print(df[['Testo', 'Emotion']].head())


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/pipelines/text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


                                               Testo  Emotion
0  Avoided this for years because people said it ...  neutral
1  When you get down to brass tacks, this still h...  disgust
2  84/100 Methodically pure sci-fi; a world with ...     fear
3  Never saw more than a few minutes of this beca...  neutral
4  the biggest blank check in Trek movie history ...      joy


In [ ]:
# Salva i risultati in un nuovo file Excel
df.to_excel('commenti_recensioni_star-trek-the-motion-picture_letterboxd_analizzati.xlsx', index=False)
